# Fish Audio Drama 3 — speech and voice cloning on a Colab Pro A100

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/Drama-3-tts-voice-clone.ipynb)

**GPU: Colab Pro A100, 40 GB or larger. A free T4 cannot run this model. An L4 cannot run the path below.**

[Fish Audio Drama 3](https://huggingface.co/fishaudio/drama-3) is a 35B-total, 3B-active mixture-of-experts text-to-speech model for multi-speaker scenes, voice cloning, and inline delivery tags such as `[sighs]`. The bf16/FP8 checkpoint is about 37 GB of engine weights. vLLM-Omni's recipe for that checkpoint asks for a single GPU with at least 48 GB. The official int4 checkpoint, [`fishaudio/drama-3-int4`](https://huggingface.co/fishaudio/drama-3-int4), keeps the routed experts and selected attention projections in 4-bit GPTQ and reports **20.8 GB of engine weights**. GPTQ-Marlin needs compute capability 8.0 or newer (A100 is 8.0; a T4 is 7.5 and cannot run it).

This notebook serves the int4 checkpoint with [SGLang-Omni](https://sgl-project.github.io/sglang-omni/cookbook/fishaudio_drama3.html), which is the runtime the model card and the int4 card both document. On a 40 GB A100 the launch budgets are tightened from the H200 defaults: engine 0.62 (~25 GB, the 20.8 GB weights plus a small cache), vocoder 0.12 (~5 GB; the codec file is 1.9 GB), preprocessing reserve 3 GiB. That is about 33 GB for one short clip. An 80 GB A100 uses the publisher's default fractions instead. Those defaults were measured on A100 80 GB and H200. A 40 GB card was not in that test matrix, so the 40 GB budget is calculated from the published weight size.

The community GGUF builds at [`Jmica/Drama-3-GGUF`](https://huggingface.co/Jmica/Drama-3-GGUF) (audio.cpp) are not used here. The Q3 file is 18.93 GiB before the FP16 audio tower, the tested Q3 setup offloads 12 expert layers to CPU, and the converter's notes say 16 GB and 24 GB GPUs were not tested. That is not a configuration this notebook can claim works on a Colab L4.

**Licence:** [Fish Audio Community License v1.0](https://huggingface.co/fishaudio/drama-3/blob/main/LICENSE), not Apache and not MIT. Research and non-commercial use is allowed. Commercial use is allowed only while annual revenue stays under USD 500,000; above that, Fish Audio requires a separate license. A hosted text-to-speech or voice-cloning service is not covered. Do not clone an identifiable person's voice without their consent. Do not use the audio to train another speech or foundation model. The reference clip below is the model's own published demo sample.

**Links:** [model card](https://huggingface.co/fishaudio/drama-3) · [int4 card](https://huggingface.co/fishaudio/drama-3-int4) · [SGLang-Omni cookbook](https://sgl-project.github.io/sglang-omni/cookbook/fishaudio_drama3.html) · [vLLM-Omni recipe](https://recipes.vllm.ai/fishaudio/drama-3)


## 1. Refuse the wrong GPU


In [ ]:
import torch

!nvidia-smi

if not torch.cuda.is_available():
    raise SystemExit("Attach a Colab Pro A100 (40 GB or larger), then rerun.")

props = torch.cuda.get_device_properties(0)
major, minor = torch.cuda.get_device_capability()
vram_gb = props.total_memory / 1024**3
print("GPU:", torch.cuda.get_device_name(), f"sm_{major}{minor}", f"{vram_gb:.1f} GB")

if major < 8:
    raise SystemExit(
        f"drama-3-int4 runs its 4-bit weights with GPTQ-Marlin, which needs compute "
        f"capability 8.0 or newer. This GPU is sm_{major}{minor}. A T4 is 7.5 and cannot run it."
    )
if vram_gb < 38:
    raise SystemExit(
        f"This notebook needs an A100 with at least 40 GB. This GPU has {vram_gb:.1f} GB. "
        "The bf16 checkpoint is ~37 GB of weights and does not fit here; the int4 engine "
        "weights are 20.8 GB and still need the vocoder and a preprocessing reserve on top."
    )


## 2. Install SGLang-Omni

The install is the cookbook command, pinned to commit `c45d6182e8d19c38408b7f580721d7c078eacf33`, into its own virtualenv so it does not replace the Colab kernel's torch. The project pins a CUDA 13 stack (`torch==2.13.0`, `sglang==0.5.21`, FlashInfer cu13, flash-attn 4). The first run downloads that stack; the server later downloads the ~22 GB int4 checkpoint. Expect both to take a while.


In [ ]:
import os
import subprocess

COMMIT = "c45d6182e8d19c38408b7f580721d7c078eacf33"
REPO = "/content/sglang-omni"
VENV = "/content/drama-venv"

subprocess.check_call(["pip", "install", "-q", "uv"])
if not os.path.isdir(REPO):
    subprocess.check_call([
        "git", "clone", "--filter=blob:none",
        "https://github.com/sgl-project/sglang-omni.git", REPO,
    ])
subprocess.check_call(["git", "fetch", "--depth", "1", "origin", COMMIT], cwd=REPO)
subprocess.check_call(["git", "checkout", "--detach", COMMIT], cwd=REPO)
print("sglang-omni", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip())

if not os.path.exists(os.path.join(VENV, "bin", "sgl-omni")):
    subprocess.check_call(["uv", "venv", VENV, "--python", "3.12"])
    subprocess.check_call([
        "uv", "pip", "install",
        "--python", os.path.join(VENV, "bin", "python"),
        "--prerelease=allow",
        "-e", ".[fishaudio]",
    ], cwd=REPO)
print("sgl-omni", os.path.join(VENV, "bin", "sgl-omni"))


## 3. Start the server

`/health` is the readiness route. The loop gives the checkpoint download and the first CUDA-graph capture up to half an hour.


In [ ]:
import time

import requests

PORT = 8000
LOG = "/tmp/drama-server.log"
SGL = os.path.join(VENV, "bin", "sgl-omni")

if vram_gb >= 80:
    budget = []
    print("80 GB or larger: publisher default memory fractions")
else:
    budget = [
        "--tts_engine.gpu_memory_fraction", "0.62",
        "--vocoder.gpu_memory_fraction", "0.12",
        "--preprocessing.total_reserve_bytes", "3GiB",
    ]
    print("40 GB budget:", " ".join(budget))

log = open(LOG, "w")
server = subprocess.Popen(
    [SGL, "serve", "--model-path", "fishaudio/drama-3-int4", "--port", str(PORT), *budget],
    stdout=log,
    stderr=subprocess.STDOUT,
    stdin=subprocess.DEVNULL,
)

base = f"http://127.0.0.1:{PORT}"
ready = False
for tick in range(120):
    if server.poll() is not None:
        break
    try:
        probe = requests.get(base + "/health", timeout=3)
        if probe.status_code == 200:
            ready = True
            print("healthy", probe.text[:200])
            break
    except requests.RequestException:
        pass
    if tick % 4 == 0:
        print(f"waiting ({tick * 15}s)")
    time.sleep(15)

if not ready:
    print(open(LOG).read()[-5000:])
    raise SystemExit(
        "sgl-omni did not become healthy. On a 40 GB A100 this budget is calculated "
        "from the 20.8 GB weight figure; the publishers measured A100 80 GB and H200. "
        "If the log is an out-of-memory error, rerun on an 80 GB A100."
    )


## 4. Text to speech

`/v1/audio/speech` with `input`, which is the cookbook's request. Sampling matches the card's published setup (temperature 1.0, top_p 0.9, top_k 30). `max_new_tokens` is capped so a short line cannot keep emitting silence until a much larger default.


In [ ]:
from IPython.display import Audio, display

SAMPLER = {"temperature": 1.0, "top_p": 0.9, "top_k": 30, "max_new_tokens": 250}


def speak(payload, filename):
    response = requests.post(base + "/v1/audio/speech", json=payload, timeout=300)
    if response.status_code != 200:
        print(response.status_code, response.text[:2000])
        response.raise_for_status()
    open(filename, "wb").write(response.content)
    print(filename, len(response.content), "bytes", response.headers.get("content-type"))
    display(Audio(filename))


speak(
    {
        "model": "fishaudio/drama-3-int4",
        "input": "Hello from Fish Audio Drama 3. This is a short text to speech demo.",
        "response_format": "wav",
        **SAMPLER,
    },
    "plain.wav",
)


## 5. Voice cloning

The reference is the 3.9 second clip the model card clones, with its transcript. Play that first, then the cloned line. The licence still forbids cloning an identifiable person without consent; this file is the published demo reference, not a voice you should swap out for someone else.


In [ ]:
REF_URL = "https://huggingface.co/datasets/zhaochenyang20/seed-tts-eval-mini/resolve/main/en/prompt-wavs/common_voice_en_10119832.wav"
REF_TEXT = "We asked over twenty different people, and they all said it was his."

ref = requests.get(REF_URL, timeout=60)
ref.raise_for_status()
open("reference.wav", "wb").write(ref.content)
print("reference clip")
display(Audio("reference.wav"))

speak(
    {
        "model": "fishaudio/drama-3-int4",
        "input": "Get the trust fund to the bank early.",
        "response_format": "wav",
        "references": [{"audio_path": REF_URL, "text": REF_TEXT}],
        **SAMPLER,
    },
    "cloned.wav",
)


## 6. Stop the server

Leave this cell until you are done listening. It frees the A100.


In [ ]:
server.terminate()
try:
    server.wait(timeout=20)
except subprocess.TimeoutExpired:
    server.kill()
log.close()
print("server stopped")
